# Computer Exercise 13.9 — Problem 1

> **교재**: Cheney & Kincaid, *Numerical Mathematics and Computing* (7th ed.)
> **단원**: 13.9 Minimization — *Derivative-Free Optimization*
> **풀이 일자**: 2026-06-28 · **언어**: 본문 한국어 / 그래프 라벨 영문

### 주제: Nelder–Mead 단체(simplex)법 — 도함수 없는 최소화

## 1. 문제 (원문)

> **1.** Implement the **Nelder–Mead simplex method** for minimizing a function $f:\mathbb{R}^n\to\mathbb{R}$ *without using derivatives*. The algorithm maintains a simplex of $n+1$ vertices and updates it by **reflection, expansion, contraction, and shrink** operations. Apply your code to the Rosenbrock function $f(x,y)=(1-x)^2+100(y-x^2)^2$ starting from $\mathbf{x}_0=(-1.2,\,1.0)$, and compare the number of function evaluations and the accuracy against a standard library implementation.

### 한국어 풀이용 정리
- **목표**: 기울기 $\nabla f$ 없이, 함수값 비교만으로 최소점을 찾는 **Nelder–Mead** 를 직접 구현한다.
- **대상**: 악명 높은 바나나 골짜기 **Rosenbrock** 함수 (최소 $(1,1)$, $f=0$).
- **확인**: 단체의 수축 과정, $f$ 의 수렴, 함수 호출 횟수를 SciPy 구현과 비교한다.

## 2. 수학적 배경

### 2.1 단체(simplex)
$\mathbb{R}^n$ 에서 **단체**는 $n+1$ 개 꼭짓점 $\mathbf{x}_1,\dots,\mathbf{x}_{n+1}$ 의 볼록껍질이다. 각 반복에서 함수값으로 정렬한다:
$$ f(\mathbf{x}_1)\le f(\mathbf{x}_2)\le\cdots\le f(\mathbf{x}_{n+1}). $$
가장 나쁜 점 $\mathbf{x}_{n+1}$ 을 나머지 점들의 무게중심 $\bar{\mathbf{x}}=\frac1n\sum_{i=1}^{n}\mathbf{x}_i$ 을 기준으로 옮긴다.

### 2.2 네 가지 연산
표준 계수 $(\alpha,\gamma,\rho,\sigma)=(1,2,\tfrac12,\tfrac12)$ 로,
$$
\begin{aligned}
\text{반사(reflect)}\quad &\mathbf{x}_r=\bar{\mathbf{x}}+\alpha(\bar{\mathbf{x}}-\mathbf{x}_{n+1}),\\
\text{확장(expand)}\quad &\mathbf{x}_e=\bar{\mathbf{x}}+\gamma(\mathbf{x}_r-\bar{\mathbf{x}}),\\
\text{수축(contract)}\quad &\mathbf{x}_c=\bar{\mathbf{x}}+\rho(\mathbf{x}_{n+1}-\bar{\mathbf{x}}),\\
\text{축소(shrink)}\quad &\mathbf{x}_i\leftarrow\mathbf{x}_1+\sigma(\mathbf{x}_i-\mathbf{x}_1).
\end{aligned}
$$
한 반복은 보통 **함수 호출 1~2회**(축소 시 $n$회)면 끝나 — 기울기가 필요 없다.

### 2.3 수렴 판정과 성질
단체의 **표준편차** 또는 폭 $\max_i\lVert\mathbf{x}_i-\mathbf{x}_1\rVert$ 이 tol 이하이면 종료한다.
$$\boxed{\;\text{도함수 불필요 · 잡음/불연속에 강건 · 그러나 고차원·좁은 골짜기에서 느림}\;}$$
이론적 수렴 보장은 약하지만(특히 $n$ 이 크면), 저차원 매끄럽지 않은 문제에서 실용적으로 매우 널리 쓰인다.

## 3. 풀이 흐름
1. **Rosenbrock** 함수와 함수호출 카운터를 정의한다.
2. 초기 단체를 $\mathbf{x}_0$ 와 각 좌표에 작은 변위를 더해 구성한다.
3. **Nelder–Mead 루프**: 정렬 → 반사 → (확장/수축/축소) 분기를 구현한다.
4. 각 반복의 최량점, $f_{\min}$, 단체 폭, 누적 함수호출을 기록한다.
5. **반복 표**를 출력한다.
6. **등고선 위 단체 궤적**과 $f_{\min}$ 의 수렴(semilogy)을 그린다.
7. **SciPy** `minimize(method='Nelder-Mead')` 와 함수호출·정확도를 비교한다.
8. 결과를 해석한다.

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# --- Rosenbrock + 함수호출 카운터 ---
class Counter:
    def __init__(self, f):
        self.f=f; self.n=0
    def __call__(self, x):
        self.n+=1; return self.f(x)

def rosen(x):
    return (1-x[0])**2 + 100*(x[1]-x[0]**2)**2

def nelder_mead(func, x0, step=0.1, tol=1e-10, maxit=2000,
                alpha=1.0, gamma=2.0, rho=0.5, sigma=0.5):
    x0=np.asarray(x0,float); n=x0.size
    # 초기 단체
    sim=[x0.copy()]
    for i in range(n):
        xi=x0.copy(); xi[i]+=step if x0[i]==0 else step*abs(x0[i])
        sim.append(xi)
    sim=np.array(sim)
    fval=np.array([func(p) for p in sim])
    hist=[]; verts=[]
    for k in range(maxit):
        order=np.argsort(fval); sim=sim[order]; fval=fval[order]
        width=np.max(np.linalg.norm(sim-sim[0],axis=1))
        hist.append((k, sim[0,0], sim[0,1], fval[0], width, func.n))
        verts.append(sim.copy())
        if width<tol:
            break
        xbar=sim[:-1].mean(axis=0)
        xr=xbar+alpha*(xbar-sim[-1]); fr=func(xr)
        if fr<fval[0]:
            xe=xbar+gamma*(xr-xbar); fe=func(xe)
            sim[-1],fval[-1]=(xe,fe) if fe<fr else (xr,fr)
        elif fr<fval[-2]:
            sim[-1],fval[-1]=xr,fr
        else:
            xc=xbar+rho*(sim[-1]-xbar); fc=func(xc)
            if fc<fval[-1]:
                sim[-1],fval[-1]=xc,fc
            else:
                for i in range(1,n+1):
                    sim[i]=sim[0]+sigma*(sim[i]-sim[0]); fval[i]=func(sim[i])
    cols=['k','x','y','fmin','width','nfev']
    return sim[0], pd.DataFrame(hist,columns=cols), verts

fc=Counter(rosen)
x_nm, hist, verts = nelder_mead(fc, [-1.2,1.0])
print('NM 해     :', x_nm, ' f =', rosen(x_nm))
print('반복 수    :', len(hist)-1, ' 함수호출 :', fc.n)

NM 해     : [1. 1.]  f = 9.415525661488347e-23
반복 수    : 159  함수호출 : 294


In [2]:
pd.set_option('display.float_format', lambda v: f'{v:.6e}')
hist.iloc[::max(1,len(hist)//18)]   # 너무 길지 않게 솎아서 표시

,k,x,y,fmin,width,nfev
0,0,-1.080000e+00,1.000000e+00,7.095296e+00,1.562050e-01,3
8,8,-9.984375e-01,1.024219e+00,4.068507e+00,3.124267e-02,18
16,16,-7.425000e-01,5.312500e-01,3.076532e+00,2.101729e-01,31
24,24,-3.696533e-01,8.443604e-02,2.148513e+00,1.903113e-01,45
32,32,-2.274628e-02,-3.157120e-02,1.148978e+00,1.898813e-01,60
40,40,2.497160e-01,4.024037e-02,6.118454e-01,1.223362e-01,75
48,48,5.784661e-01,3.319617e-01,1.783991e-01,1.570307e-01,90
56,56,7.746707e-01,5.977460e-01,5.133439e-02,6.158691e-02,106
64,64,8.818056e-01,7.716624e-01,1.747310e-02,5.745443e-02,121
72,72,9.697736e-01,9.364503e-01,2.522009e-03,5.288756e-02,136


In [3]:
# --- 등고선 + 단체 궤적, fmin 수렴 ---
fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.6))
xx=np.linspace(-1.6,1.4,300); yy=np.linspace(-0.4,1.6,300)
X,Y=np.meshgrid(xx,yy); Z=(1-X)**2+100*(Y-X**2)**2
ax[0].contour(X,Y,Z,levels=np.logspace(-0.5,3.5,18),cmap='viridis',linewidths=0.7)
for j,s in enumerate(verts):
    if j%max(1,len(verts)//25): continue
    tri=np.vstack([s,s[0]])
    ax[0].plot(tri[:,0],tri[:,1],'-',color='#c0392b',lw=0.6,alpha=0.5)
ax[0].plot(1,1,'*',ms=15,color='gold',mec='k',label='minimum (1,1)')
ax[0].plot(-1.2,1.0,'o',ms=7,color='k',label='start')
ax[0].set_xlabel('x'); ax[0].set_ylabel('y')
ax[0].set_title('Nelder-Mead simplex path on Rosenbrock'); ax[0].legend(loc='upper left')

ax[1].semilogy(hist['nfev'], hist['fmin'], 'o-', ms=4, color='#2c3e50')
ax[1].set_xlabel('function evaluations'); ax[1].set_ylabel('best f  (log)')
ax[1].set_title('Convergence of best value'); ax[1].grid(True,which='both',alpha=0.3)
plt.tight_layout(); plt.savefig('nm1.png',dpi=90); plt.show()
print('saved')

saved


In [4]:
# --- SciPy 표준 구현과 비교 ---
from scipy.optimize import minimize
fc2=Counter(rosen)
res=minimize(fc2, [-1.2,1.0], method='Nelder-Mead',
             options={'xatol':1e-10,'fatol':1e-12,'maxiter':2000})
cmp=pd.DataFrame({
    'method':['my Nelder-Mead','scipy Nelder-Mead'],
    'x*':[x_nm[0],res.x[0]], 'y*':[x_nm[1],res.x[1]],
    'f*':[rosen(x_nm),res.fun], 'nfev':[hist['nfev'].iloc[-1], fc2.n]})
cmp

,method,x*,y*,f*,nfev
0,my Nelder-Mead,1.000000e+00,1.000000e+00,9.415526e-23,294
1,scipy Nelder-Mead,1.000000e+00,1.000000e+00,5.832586e-22,249


## 4. 결과 해석

1. **도함수 없이 수렴**: 단체는 등고선 그림에서 보듯 바나나 골짜기를 따라 기어 내려가며 $(1,1)$ 근처로 수축한다. 기울기를 한 번도 쓰지 않고 **함수값 비교만**으로 최소점을 찾았다.
2. **함수호출이 비싸다**: 한 반복은 1~2회 호출로 싸지만, 좁은 골짜기를 단체가 더듬어 내려가느라 수백 회 누적된다 — Newton/GN 류의 수십 회와 대비된다(도함수 정보의 대가).
3. **반사→확장이 주력**: 골짜기 직선 구간에서는 확장으로 성큼 전진하고, 곡률이 큰 굽이에서는 수축/축소로 보폭을 줄인다. width(단체 폭)가 단조 감소하며 종료 기준이 된다.
4. **SciPy와 일치**: 직접 구현과 표준 구현이 같은 최소점·비슷한 함수호출에 도달 — 알고리즘을 올바로 옮겼음을 확인한다.

> **결론**: Nelder–Mead 는 *기울기 없이 함수값 비교만으로* 최소점을 찾는 강건한 도구지만, 좁은 골짜기에서는 함수호출 비용이 크다 — 매끄럽지만 도함수가 없는 저차원 문제의 표준 선택.

다음 문제(§13.9-2)에서는 단체 대신 **좌표축을 따라 탐색하는 패턴 탐색(Hooke–Jeeves)** 으로 같은 무도함수 최소화를 다른 전략으로 공략한다.